# Kurgans UNet Experiments on Kaggle

Notebook only prepares Kaggle paths, launches `02_unet_segmentation/run_kaggle_experiments.sh`, previews outputs, and archives runs.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get(
    'REPO_URL',
    'https://github.com/MataNerdy/Geodata_Archaeology_CV.git',
)
BRANCH = os.environ.get('BRANCH', 'main')

REPO_DIR = Path('/kaggle/working/Geodata_Archaeology_CV')
WORK_REPO = REPO_DIR
DATA_ROOT = Path('/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset')
SEG_DIR = WORK_REPO / '02_unet_segmentation'
RUN_ROOT = SEG_DIR / 'runs'
BASELINE_DIR = RUN_ROOT / 'baseline_all_modalities_ce_dice'

print('Repo URL:', REPO_URL)
print('Branch:', BRANCH)
print('Work repo:', WORK_REPO)
print('Data root:', DATA_ROOT)


## GPU Check

In [ ]:
!nvidia-smi || true

import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## Clone Or Update Repository

In [ ]:
if REPO_DIR.exists():
    print('Repo already exists. Pulling latest changes...')
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
else:
    print('Cloning repo...')
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    os.chdir(REPO_DIR)

PROJECT_DIR = REPO_DIR
sys.path.insert(0, str(PROJECT_DIR))

print(f'Using project directory: {PROJECT_DIR}')
print('Segmentation directory exists:', SEG_DIR.exists())
if not SEG_DIR.exists():
    raise FileNotFoundError(SEG_DIR)

subprocess.run(['git', 'status'], check=True)
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)


## Dataset Check

In [ ]:
required = [
    DATA_ROOT,
    DATA_ROOT / 'metadata.csv',
    DATA_ROOT / 'images',
    DATA_ROOT / 'masks',
    SEG_DIR / 'run_kaggle_experiments.sh',
]
for path in required:
    print(path, 'OK' if path.exists() else 'MISSING')
    if not path.exists():
        raise FileNotFoundError(path)

print('Images:', len(list((DATA_ROOT / 'images').glob('*.npy'))))
print('Masks:', len(list((DATA_ROOT / 'masks').glob('*.npy'))))

## Run Experiments

In [ ]:
env = os.environ.copy()
env['DATA_ROOT'] = str(DATA_ROOT)
env['RUN_ROOT'] = str(RUN_ROOT)

subprocess.run(
    ['bash', 'run_kaggle_experiments.sh'],
    cwd=SEG_DIR,
    env=env,
    check=True,
)

## History

In [ ]:
import pandas as pd

history_path = BASELINE_DIR / 'history.csv'
history = pd.read_csv(history_path)
display(history.tail())

## Prediction Examples

In [ ]:
from IPython.display import Image, display

prediction_path = BASELINE_DIR / 'prediction_examples.png'
if not prediction_path.exists():
    prediction_path = BASELINE_DIR / 'prediction_examples_eval.png'

display(Image(filename=str(prediction_path)))

## Archive Runs

In [ ]:
from IPython.display import FileLink, display

archive_base = Path('/kaggle/working/kurgans_runs')
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        'zip',
        root_dir=RUN_ROOT.parent,
        base_dir=RUN_ROOT.name,
    )
)
print('Created archive:', archive_path)
display(FileLink(str(archive_path)))